# 69 — Latency Profiling
**Goal:** Profile pipeline latency — find bottlenecks in parsing, embedding, and LLM calls.

## 1. Why Latency Matters

In [ ]:
print('''Resume analysis pipeline latency budget:
- PDF parsing: ~200ms (pdfplumber)
- OCR: ~2-5s (if needed)
- Section detection: ~10ms
- Skill extraction (regex): ~5ms
- Skill normalization: ~50ms
- Embedding: ~100ms (sentence-transformers)
- LLM rewriting: ~1-3s (gpt-4o-mini)
- FAISS search: ~10ms (1M vectors)

Total: ~300ms (no LLM) or ~2-4s (with LLM)
Bottlenecks: OCR, LLM calls, embedding generation''')

## 2. Simple Profiling

In [ ]:
import time

def profile_pipeline(text):
    """Profile each stage of the pipeline."""
    times = {}
    
    # Stage 1: Text normalization
    t0 = time.time()
    normalized = text.lower().strip()
    times['normalization'] = time.time() - t0
    
    # Stage 2: Section detection (simulated)
    t0 = time.time()
    sections = {"summary": True, "skills": True}
    time.sleep(0.005)  # Simulate work
    times['section_detect'] = time.time() - t0
    
    # Stage 3: Skill extraction (regex)
    t0 = time.time()
    import re
    skills = re.findall(r"\\b(Python|Java|SQL|AWS|NLP)\\b", text, re.IGNORECASE)
    times['skill_extract'] = time.time() - t0
    
    # Stage 4: Embedding (simulated)
    t0 = time.time()
    time.sleep(0.05)  # Simulate 50ms embedding
    times['embedding'] = time.time() - t0
    
    total = sum(times.values())
    print(f"Pipeline profile ({total*1000:.0f}ms total):")
    for stage, t in times.items():
        pct = t / total * 100
        print(f"  {stage:20s} {t*1000:6.1f}ms ({pct:.0%})")
    return times

text = "Python developer with 5 years NLP and AWS experience"
profile_pipeline(text)

## 3. Profiling with cProfile

In [ ]:
import cProfile, pstats, io

def slow_function():
    """A slow function we want to profile."""
    result = []
    for i in range(1000):
        result.append(sum(range(i * 100)))
    return result

# Profile
profiler = cProfile.Profile()
profiler.enable()
slow_function()
profiler.disable()

s = io.StringIO()
ps = pstats.Stats(profiler, stream=s).sort_stats('cumtime')
ps.print_stats(10)
print("cProfile output (top 10 by cumulative time):")
print(s.getvalue())

## 4. Optimizing Bottlenecks

In [ ]:
print('''Bottleneck optimization strategies:
1. PDF parsing:
   - Cache parsed text by file hash
   - Use PyMuPDF (faster than pdfplumber)
   - Parallel parse multiple documents

2. Embedding:
   - Precompute and cache all resume embeddings
   - Use smaller models (MiniLM vs MPNet)
   - Batch encode (model.encode(list_of_texts))

3. LLM calls:
   - Use streaming for real-time apps
   - Set low max_tokens for extraction tasks
   - Cache common queries
   - Use cheaper models for simple tasks''')

## Summary: Profile before optimizing. Target the biggest bottleneck first. Cache aggressively.